<a href="https://colab.research.google.com/github/Srikanth5414/Aspect-Based-Sentiment-Analysis/blob/main/kaggle5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import os

# Upload file(s)
uploaded = files.upload()

# List uploaded files
for filename in uploaded.keys():
    print(f'File uploaded: {filename}')

Saving Laptop_Train_v2.csv to Laptop_Train_v2 (1).csv
File uploaded: Laptop_Train_v2 (1).csv


In [ ]:
# Core dependencies
!pip install emoji imbalanced-learn
!pip install seaborn matplotlib tensorflow

# NLTK (for tokenization/stopwords)
import nltk
nltk.download('punkt')
nltk.download('stopwords')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 9.9 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# Data handling
import numpy as np
import pandas as pd

# Deep learning
import tensorflow as tf

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text processing
import re, string
import emoji
import nltk

# ML tools
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Handling imbalance
from imblearn.over_sampling import RandomOverSampler

In [ ]:
df = pd.read_csv('/content/Laptop_Train_v2.csv')

In [ ]:
print(df.head())

     id                                           Sentence     Aspect Term  \
0  2339  I charge it at night and skip taking the cord ...            cord   
1  2339  I charge it at night and skip taking the cord ...    battery life   
2  1316  The tech guy then said the service center does...  service center   
3  1316  The tech guy then said the service center does...    "sales" team   
4  1316  The tech guy then said the service center does...        tech guy   

   polarity  from   to  
0   neutral    41   45  
1  positive    74   86  
2  negative    27   41  
3  negative   109  121  
4   neutral     4   12  


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2358 entries, 0 to 2357
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           2358 non-null   int64 
 1   Sentence     2358 non-null   object
 2   Aspect Term  2358 non-null   object
 3   polarity     2358 non-null   object
 4   from         2358 non-null   int64 
 5   to           2358 non-null   int64 
dtypes: int64(3), object(3)
memory usage: 110.7+ KB


In [ ]:
!pip install contractions
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions

# Download resources (run once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# ---- Drop conflict rows ----
df = df[df['polarity'] != 'conflict']
print("After dropping conflicts:", df.shape)

# Function to clean text
def clean_text(text):
    # Ensure input is a string
    text = str(text)

    # Expand contractions
    text = contractions.fix(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove digits
    text = re.sub(r'\d+', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize and remove stopwords + lemmatize
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

# Apply cleaning to 'Sentence' column
df['clean_text'] = df['Sentence'].apply(clean_text)

# Clean aspect terms as well
df['aspect_clean'] = df['Aspect Term'].apply(clean_text)

# Encode polarity labels (if you plan for ML/DL)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['polarity_encoded'] = le.fit_transform(df['polarity'])

print(df[['Sentence', 'Aspect Term', 'polarity', 'clean_text', 'aspect_clean', 'polarity_encoded']].head())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.9/113.9 kB 9.0 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


After dropping conflicts: (2313, 6)
                                            Sentence     Aspect Term  \
0  I charge it at night and skip taking the cord ...            cord   
1  I charge it at night and skip taking the cord ...    battery life   
2  The tech guy then said the service center does...  service center   
3  The tech guy then said the service center does...    "sales" team   
4  The tech guy then said the service center does...        tech guy   

   polarity                                         clean_text  \
0   neutral    charge night skip taking cord good battery life   
1  positive    charge night skip taking cord good battery life   
2  negative  tech guy said service center exchange direct c...   
3  negative  tech guy said service center exchange direct c...   
4   neutral  tech guy said service center exchange direct c...   

     aspect_clean  polarity_encoded  
0            cord                 1  
1    battery life                 2  
2  service center   

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

# Splitting the data into training, validation, and testing sets
X = df[['clean_text', 'aspect_clean']]
y = df['polarity_encoded']

# First split: train + test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: train -> train + val
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

print("Shapes before oversampling:")
print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

# Oversampling the training data to handle class imbalance
ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(X_train, y_train)

print("Shapes after oversampling:")
print("Train:", X_train_resampled.shape, y_train_resampled.shape)


Shapes before oversampling:
Train: (1665, 2) (1665,)
Val: (185, 2) (185,)
Test: (463, 2) (463,)
Shapes after oversampling:
Train: (2130, 2) (2130,)


In [ ]:
from transformers import BertTokenizer
import numpy as np

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_bert(texts, max_len):
    input_ids, attention_masks = [], []
    for txt in texts.tolist():   # ensure it's a list of strings
        encoded = tokenizer.encode_plus(
            txt,
            add_special_tokens=True,     # [CLS], [SEP]
            max_length=max_len,
            truncation=True,
            padding='max_length',
            return_attention_mask=True,
        )
        input_ids.append(encoded['input_ids'])
        attention_masks.append(encoded['attention_mask'])
    return np.array(input_ids), np.array(attention_masks)

# Example usage
X_train_ids, X_train_mask = tokenize_bert(X_train_resampled['clean_text'], max_len=50)
X_val_ids,   X_val_mask   = tokenize_bert(X_val['clean_text'], max_len=50)
X_test_ids,  X_test_mask  = tokenize_bert(X_test['clean_text'], max_len=50)

print("Train IDs shape:", X_train_ids.shape)
print("Validation IDs shape:", X_val_ids.shape)
print("Test IDs shape:", X_test_ids.shape)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Train IDs shape: (2130, 50)
Validation IDs shape: (185, 50)
Test IDs shape: (463, 50)


In [ ]:
from keras.utils import to_categorical

# One-hot encode the target variable
y_train_one_hot = to_categorical(y_train_resampled)
y_val_one_hot = to_categorical(y_val)
y_test_one_hot = to_categorical(y_test)

In [ ]:
import tensorflow as tf

train_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': X_train_ids,
        'attention_mask': X_train_mask
    },
    y_train_one_hot  # your labels
)).batch(32)

val_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': X_val_ids,
        'attention_mask': X_val_mask
    },
    y_val_one_hot
)).batch(32)

In [ ]:
from transformers import TFBertForSequenceClassification

num_labels = 3  # e.g. Positive, Negative, Neutral

model = TFBertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
df.head()

,id,Sentence,Aspect Term,polarity,from,to,clean_text,aspect_clean,polarity_encoded
0,2339,I charge it at night and skip taking the cord ...,cord,neutral,41,45,charge night skip taking cord good battery life,cord,1
1,2339,I charge it at night and skip taking the cord ...,battery life,positive,74,86,charge night skip taking cord good battery life,battery life,2
2,1316,The tech guy then said the service center does...,service center,negative,27,41,tech guy said service center exchange direct c...,service center,0
3,1316,The tech guy then said the service center does...,"""sales"" team",negative,109,121,tech guy said service center exchange direct c...,sale team,0
4,1316,The tech guy then said the service center does...,tech guy,neutral,4,12,tech guy said service center exchange direct c...,tech guy,1


In [ ]:
# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.CategoricalAccuracy("accuracy")

model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5
)

Epoch 1/5
67/67 [==============================] - 1219s 18s/step - loss: 0.9736 - accuracy: 0.5249 - val_loss: 1.2588 - val_accuracy: 0.2000
Epoch 2/5
67/67 [==============================] - 1152s 17s/step - loss: 0.9446 - accuracy: 0.4423 - val_loss: 1.5099 - val_accuracy: 0.2649
Epoch 3/5
67/67 [==============================] - 1146s 17s/step - loss: 0.8741 - accuracy: 0.5113 - val_loss: 1.3158 - val_accuracy: 0.3514
Epoch 4/5
67/67 [==============================] - 1145s 17s/step - loss: 0.7819 - accuracy: 0.5775 - val_loss: 1.1523 - val_accuracy: 0.5081
Epoch 5/5
67/67 [==============================] - 1146s 17s/step - loss: 0.6600 - accuracy: 0.6746 - val_loss: 1.0103 - val_accuracy: 0.5243


In [ ]:
from transformers import BertTokenizer
import tensorflow as tf

# Load tokenizer (same as training)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Example sentence + aspect
sentence = "The battery life is too short but the screen is great."
aspect = "battery life"

# Encode in ABSA format: [CLS] sentence [SEP] aspect [SEP]
encoded = tokenizer.encode_plus(
    sentence,
    aspect,
    add_special_tokens=True,
    max_length=50,
    truncation=True,
    padding="max_length",
    return_attention_mask=True,
    return_tensors="tf"
)

input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

# Predict
logits = model.predict({"input_ids": input_ids, "attention_mask": attention_mask}).logits
probs = tf.nn.softmax(logits, axis=-1).numpy()
pred_label = probs.argmax(axis=-1)[0]

label_map = { 0: "negative", 1:"neutral", 2: "positive"}

print("Sentence:", sentence)
print("Aspect:", aspect)
print("Logits:", logits)
print("Probabilities:", probs)
print("Predicted Label:", label_map[pred_label])

1/1 [==============================] - 11s 11s/step
Sentence: The battery life is too short but the screen is great.
Aspect: battery life
Logits: [[ 0.6147238   0.53876555 -1.0438036 ]]
Probabilities: [[0.47230542 0.4377586  0.08993602]]
Predicted Label: negative


In [ ]:
from transformers import BertTokenizer
import tensorflow as tf

# Load tokenizer (same as training)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Example from Laptop dataset
sentence = "The laptop performance is amazing but the battery drains quickly."
aspect = "laptop"

# Encode in ABSA format
encoded = tokenizer.encode_plus(
    sentence,
    aspect,
    add_special_tokens=True,
    max_length=50,
    truncation=True,
    padding="max_length",
    return_attention_mask=True,
    return_tensors="tf"
)

# Predict
outputs = model.predict(
    {"input_ids": encoded["input_ids"], "attention_mask": encoded["attention_mask"]},
    verbose=0
)

logits = outputs.logits
probs = tf.nn.softmax(logits, axis=-1).numpy()
pred_label = probs.argmax(axis=-1)[0]

label_map = { 0: "negative", 1: "neutral", 2: "positive"}

print("Sentence:", sentence)
print("Aspect:", aspect)
print("Probabilities:", probs)
print("Predicted Label:", label_map[pred_label])


Sentence: The laptop performance is amazing but the battery drains quickly.
Aspect: laptop
Probabilities: [[0.28830627 0.5678611  0.1438326 ]]
Predicted Label: neutral


In [ ]:
# Example positive sentence + aspect
sentence = "The camera quality is amazing and takes stunning pictures."
aspect = "camera quality"

# Encode in ABSA format: [CLS] sentence [SEP] aspect [SEP]
encoded = tokenizer.encode_plus(
    sentence,
    aspect,
    add_special_tokens=True,
    max_length=50,
    truncation=True,
    padding="max_length",
    return_attention_mask=True,
    return_tensors="tf"
)

input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

# Predict
logits = model.predict({"input_ids": input_ids, "attention_mask": attention_mask}).logits
probs = tf.nn.softmax(logits, axis=-1).numpy()
pred_label = probs.argmax(axis=-1)[0]

label_map = {0: "negative", 1: "neutral", 2: "positive"}

print("Sentence:", sentence)
print("Aspect:", aspect)
print("Logits:", logits)
print("Probabilities:", probs)
print("Predicted Label:", label_map[pred_label])


1/1 [==============================] - 0s 337ms/step
Sentence: The camera quality is amazing and takes stunning pictures.
Aspect: camera quality
Logits: [[-1.6926982  -0.39519635  2.3673475 ]]
Probabilities: [[0.01596498 0.05843408 0.9256009 ]]
Predicted Label: positive


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Convert one-hot encoded y_test to class indices
y_test_labels = np.argmax(y_test_one_hot, axis=1)

# Convert model predictions (logits) to class indices
y_pred_bert = model.predict({"input_ids": X_test_ids, "attention_mask": X_test_mask}).logits
y_pred_labels = np.argmax(y_pred_bert, axis=1)

# Print classification report
print('\tClassification Report for BERT:\n\n',
      classification_report(y_test_labels, y_pred_labels,
                            target_names=['Negative', 'Neutral', 'Positive']))


15/15 [==============================] - 83s 5s/step
	Classification Report for BERT:

               precision    recall  f1-score   support

    Negative       0.78      0.46      0.58       173
     Neutral       0.30      0.85      0.45        92
    Positive       0.87      0.45      0.60       198

    accuracy                           0.54       463
   macro avg       0.65      0.59      0.54       463
weighted avg       0.72      0.54      0.56       463



In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict
result_bert = model.predict({"input_ids": X_test_ids, "attention_mask": X_test_mask}).logits

# Convert logits -> class indices
y_pred_labels = np.argmax(result_bert, axis=1)

# Convert one-hot test labels -> class indices
y_test_labels = np.argmax(y_test_one_hot, axis=1)

# Classification report
print('\tClassification Report for BERT:\n\n',
      classification_report(y_test_labels, y_pred_labels,
                            target_names=['Negative', 'Neutral', 'Positive']))


15/15 [==============================] - 81s 5s/step
	Classification Report for BERT:

               precision    recall  f1-score   support

    Negative       0.78      0.46      0.58       173
     Neutral       0.30      0.85      0.45        92
    Positive       0.87      0.45      0.60       198

    accuracy                           0.54       463
   macro avg       0.65      0.59      0.54       463
weighted avg       0.72      0.54      0.56       463

